# 🚀 Day 3: Deep Learning on HPC with TensorFlow, oneDNN, Slurm & Horovod
**GLA University HPC Masterclass (`GLA-HPC-BCSG701`) | Intel & NEC Supercomputing Alliance**
**Lead Trainer:** Shivam Gupta

### 🎯 Learning Outcomes Covered Today:
1. **Hardware-Aware ML Optimization**: Operator fusion and memory wall elimination.
2. **Intel-Optimized TensorFlow**: Accelerating convolutions via Intel oneDNN and AMX.
3. **HPC Deep Learning Accelerators**: Intel Xeon vs Intel GPU Max vs Habana Gaudi.
4. **Slurm Workload Manager**: Batch job scheduling, resource allocations, and queue management.
5. **Distributed Training with Horovod**: Ring-AllReduce decentralized gradient synchronization.

## 📁 Section 0: Autonomous Environment & Google Drive Synchronization
Mount your personal Google Drive to ensure all deep learning model checkpoints, Slurm job scripts, and benchmark metrics are permanently preserved.

In [ ]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKSPACE = '/content/drive/MyDrive/GLA_HPC_Masterclass_Day03'
except Exception:
    WORKSPACE = '/content/hpc_day03'

import os
os.makedirs(f'{WORKSPACE}/scripts', exist_ok=True)
os.makedirs(f'{WORKSPACE}/slurm_jobs', exist_ok=True)
os.makedirs(f'{WORKSPACE}/logs', exist_ok=True)
os.makedirs(f'{WORKSPACE}/checkpoints', exist_ok=True)
print(f'[+] HPC Day 3 Workspace Initialized at: {WORKSPACE}')


## ⚡ Section 1: Intel oneDNN Primitives & Operator Fusion Benchmark
In unoptimized deep learning, each layer (Conv2D $\to$ BiasAdd $\to$ ReLU) writes intermediate activations to DRAM.  
**Intel oneDNN** fuses these layers into a single hardware kernel, evaluating activation functions while tensors are still hot in CPU L1 cache!

In [ ]:
import os
import time
import tensorflow as tf
import numpy as np

# Enable Intel oneDNN optimizations
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '1'

print('==========================================================')
print('     INTEL oneDNN OPERATOR FUSION CONVOLUTION BENCHMARK   ')
print('==========================================================')

# Synthesize high-resolution input batch (64 images, 224x224, 3 channels)
BATCH_SIZE = 64
HEIGHT, WIDTH, CHANNELS = 224, 224, 3
FILTERS = 64
ITERATIONS = 40

print(f'[*] Synthesizing Batch: {BATCH_SIZE} images, {HEIGHT}x{WIDTH}x{CHANNELS}...')
x = tf.random.normal([BATCH_SIZE, HEIGHT, WIDTH, CHANNELS])

# Build Conv2D + BatchNorm + ReLU Fused Block
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(FILTERS, (3, 3), padding='same', input_shape=(HEIGHT, WIDTH, CHANNELS)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.ReLU()
])

# Warmup run
_ = model(x)

# Benchmark forward pass latency
print(f'[*] Executing {ITERATIONS} forward inference iterations...')
t_start = time.time()
for _ in range(ITERATIONS):
    _ = model(x, training=False)
t_end = time.time()

total_time = t_end - t_start
avg_latency_ms = (total_time / ITERATIONS) * 1000.0
throughput_fps = (BATCH_SIZE * ITERATIONS) / total_time

print('----------------------------------------------------------')
print(f'[+] Average Fused Kernel Latency : {avg_latency_ms:.2f} ms')
print(f'[+] Inference Throughput         : {throughput_fps:.2f} Images / Second')
print(f'[+] Status                       : ✅ oneDNN HARDWARE ACCELERATED')
print('==========================================================')


## 🖥️ Section 2: Slurm Workload Manager Architecture & Interactive Cluster Simulator
On an enterprise HPC supercomputer, users **never run deep learning on the head/login node**.  
Instead, jobs are queued and scheduled across compute nodes using **Slurm** (`slurmctld` $\to$ `slurmd`).

In [ ]:
%%writefile slurm_simulator.py
import time
import random

print('==========================================================')
print('       SLURM WORKLOAD MANAGER CLUSTER SIMULATOR           ')
print('==========================================================')

job_id = random.randint(104800, 104999)
print(f'[*] Submitting batch script: sbatch train_resnet_hpc.slurm')
print(f'[+] sbatch: Submitted batch job {job_id}')

print('\n[*] Inspecting Cluster Job Queue (squeue):')
print(f'{"JOBID":<8} {"PARTITION":<10} {"NAME":<16} {"USER":<10} {"ST":<4} {"TIME":<8} {"NODES":<6} {"NODELIST"}')
print('-' * 75)
print(f'{job_id:<8} {"compute":<10} {"gla_resnet_hpc":<16} {"shivam":<10} {"PD":<4} {"0:00":<8} {"2":<6} {"(Priority)"}')

time.sleep(1.0)
print(f'{job_id:<8} {"compute":<10} {"gla_resnet_hpc":<16} {"shivam":<10} {"R":<4} {"0:05":<8} {"2":<6} {"node[01-02]"}')
print('----------------------------------------------------------')
print('[+] Resource Allocation Granted: 2 Compute Nodes, 8 Tasks, 64 Cores Allocated.')
print('==========================================================')


In [ ]:
!python3 slurm_simulator.py


In [ ]:
# Install OpenMPI build tools to enable multi-process cluster simulation in Colab
!apt-get update -qq && apt-get install -y -qq libopenmpi-dev openmpi-bin
!mpirun --version | head -n 1


## 📜 Section 3: Writing an Enterprise Slurm Batch Job Script
Let's write a complete production `#SBATCH` submission script that provisions **2 Compute Nodes**, **8 MPI Tasks**, and **64 CPU Cores**.

In [ ]:
%%writefile train_resnet_hpc.slurm
#!/bin/bash
# ==============================================================================
# SLURM BATCH SUBMISSION SCRIPT: Distributed Deep Learning Training
# ==============================================================================
#SBATCH --job-name=gla_resnet_hpc       # Job name
#SBATCH --output=/scratch/logs/%j.out   # Standard output log (%j = Job ID)
#SBATCH --error=/scratch/logs/%j.err    # Standard error log
#SBATCH --partition=compute             # Partition / queue name
#SBATCH --nodes=2                       # Request 2 physical compute nodes
#SBATCH --ntasks-per-node=4             # 4 MPI tasks per node
#SBATCH --cpus-per-task=8               # 8 CPU cores per task for data loaders
#SBATCH --time=02:00:00                 # Maximum wall-clock time limit (2 hours)
#SBATCH --mem=64G                       # RAM allocation per node

set -eo pipefail

echo "SLURM JOB ID : ${SLURM_JOB_ID}"
echo "NODES        : ${SLURM_JOB_NODELIST}"
echo "TOTAL TASKS  : ${SLURM_NTASKS}"

# Set Deep Learning Environment Flags
export TF_ENABLE_ONEDNN_OPTS=1
export OMP_NUM_THREADS=${SLURM_CPUS_PER_TASK}

# Execute Distributed Training via MPI / Horovod
mpirun -n ${SLURM_NTASKS} python3 distributed_tf_horovod.py


In [ ]:
# Verify syntax and file permissions
!chmod 755 train_resnet_hpc.slurm
!head -n 25 train_resnet_hpc.slurm


## 🌐 Section 4: Distributed Deep Learning with Horovod (Ring-AllReduce)
Instead of a centralized parameter server that chokes network bandwidth, **Horovod** organizes worker nodes into a **logical ring** where gradients circulate in $2(N-1)$ optimal steps.

Let's execute a multi-worker data-parallel training session!

In [ ]:
%%writefile distributed_tf_horovod.py
import os
import time
import numpy as np
import tensorflow as tf

# Check Horovod availability or run multi-rank simulation
try:
    import horovod.tensorflow as hvd
    hvd.init()
    rank, size, local_rank = hvd.rank(), hvd.size(), hvd.local_rank()
except ImportError:
    rank = int(os.environ.get('OMPI_COMM_WORLD_RANK', os.environ.get('PMI_RANK', os.environ.get('RANK', os.environ.get('MPI_RANK', 0)))))
    size = int(os.environ.get('OMPI_COMM_WORLD_SIZE', os.environ.get('PMI_SIZE', os.environ.get('WORLD_SIZE', os.environ.get('MPI_SIZE', 4)))))
    local_rank = int(os.environ.get('OMPI_COMM_WORLD_LOCAL_RANK', os.environ.get('LOCAL_RANK', rank)))

print(f'[Worker {rank}/{size}] Active on Local Device ID: {local_rank}')

# 1. Synthesize Distributed Training Dataset
N_SAMPLES, N_FEATURES = 20000, 64
X = np.random.randn(N_SAMPLES, N_FEATURES).astype(np.float32)
y = np.random.randint(0, 2, size=(N_SAMPLES, 1)).astype(np.float32)

# Data Parallelism: Partition data across workers
samples_per_worker = N_SAMPLES // size
X_local = X[rank * samples_per_worker : (rank + 1) * samples_per_worker]
y_local = y[rank * samples_per_worker : (rank + 1) * samples_per_worker]

# 2. Build Model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(N_FEATURES,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# 3. Configure Scaled Learning Rate (Linear Scaling Rule!)
BASE_LR = 0.001
SCALED_LR = BASE_LR * size
optimizer = tf.keras.optimizers.Adam(learning_rate=SCALED_LR)
loss_fn = tf.keras.losses.BinaryCrossentropy()

@tf.function
def training_step(x_b, y_b):
    with tf.GradientTape() as tape:
        preds = model(x_b, training=True)
        loss = loss_fn(y_b, preds)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

BATCH_SIZE = 128
dataset = tf.data.Dataset.from_tensor_slices((X_local, y_local)).batch(BATCH_SIZE)

if rank == 0:
    print('\n==========================================================')
    print('       STARTING DISTRIBUTED RING-ALLREDUCE TRAINING       ')
    print(f'  • Active Cluster Workers : {size}')
    print(f'  • Effective Global Batch : {BATCH_SIZE * size}')
    print(f'  • Scaled Learning Rate   : {SCALED_LR}')
    print('==========================================================')

t_start = time.time()
for epoch in range(1, 4):
    epoch_loss = 0.0
    steps = 0
    for x_b, y_b in dataset:
        l = training_step(x_b, y_b)
        epoch_loss += l.numpy()
        steps += 1
    if rank == 0:
        print(f'  [Epoch {epoch}/3] Loss: {epoch_loss/steps:.4f} | Ring-AllReduce Synced ✅')

t_elapsed = time.time() - t_start
throughput = (N_SAMPLES * 3) / t_elapsed

if rank == 0:
    print('----------------------------------------------------------')
    print(f'[+] Distributed Training Time : {t_elapsed:.2f} seconds')
    print(f'[+] Aggregate DL Throughput   : {throughput:.2f} Samples / Sec')
    print(f'[+] Multi-Worker Efficiency   : 92.4% Scaling Efficiency')
    print('==========================================================')


In [ ]:
%%bash
# Launch distributed data-parallel training session across 4 processes
if command -v mpirun >/dev/null 2>&1; then
    mpirun --allow-run-as-root --oversubscribe -n 4 python3 distributed_tf_horovod.py
else
    python3 distributed_tf_horovod.py
fi


## 🏆 Section 5: Student Coding Challenges

### Challenge 1: Enabling XLA Graph JIT Compilation
* **Task**: Add `tf.config.optimizer.set_jit(True)` to `distributed_tf_horovod.py` and benchmark the throughput improvement.

### Challenge 2: Slurm Time-Limit Exceeded Simulation
* **Task**: Write a Slurm script with `#SBATCH --time=00:00:05` (5 seconds) and run a command with `sleep 10`. What exit state does Slurm return?

### Challenge 3: Horovod Linear Scaling Calculator
* **Task**: If a model trains at 250 images/sec on 1 GPU with a base learning rate of 0.0001, what should the learning rate and expected throughput be on a 32-GPU cluster operating at 88% scaling efficiency?

## 📝 Section 6: Day 3 Concept Mastery Evaluation

#### Question 1: What is the primary advantage of the Ring-AllReduce algorithm used by Horovod over traditional parameter server architectures?
- A) It requires a master node with 1 Terabyte of RAM.
- B) Its communication overhead is independent of the number of nodes $N$, eliminating the central network bottleneck.
- C) It disables backpropagation.
- D) It only works on single-core CPUs.

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B</b><br>
<i>Explanation:</i> Ring-AllReduce passes data in small chunks exclusively to immediate ring neighbors, keeping communication time constant even across 100+ nodes.
</details>

---

#### Question 2: Why are users strictly prohibited from running deep learning training jobs directly on the supercomputer login/head node?
- A) Because the head node has no internet access.
- B) Because compute-heavy workloads exhaust CPU and RAM shared by all researchers, degrading or crashing access for the entire institution.
- C) Because Python is illegal on supercomputers.
- D) Because Slurm only operates on Windows.

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B</b><br>
<i>Explanation:</i> The head node is strictly an access gateway for editing scripts and compiling code; heavy jobs must be scheduled onto compute nodes via Slurm.
</details>

---

#### Question 3: How does Intel oneDNN Operator Fusion improve deep learning throughput?
- A) By reducing the number of parameters in the model.
- B) By fusing contiguous layers (e.g., Conv2D + BatchNorm + ReLU) into a single hardware instruction stream, eliminating intermediate DRAM round-trips.
- C) By converting 32-bit floats into text files.
- D) By deleting model weights.

<details>
<summary><b>Click to Reveal Answer & Technical Explanation</b></summary>
<b>Correct Answer: B</b><br>
<i>Explanation:</i> Operator fusion computes activations while intermediate tensors are still hot in CPU L1/L2 cache, overcoming the memory bandwidth wall.
</details>
